# RentaSmart · Fase 3: exploración, filtros y modelado

Notebook reproducible para explorar rentas anunciadas en México y evaluar un modelo de referencia para **Ciudad de México, alquiler, casas y departamentos**. La fuente original contiene agregados por ciudad, zona, operación y tipo: **cada fila no equivale a una vivienda individual**.

**Fuente:** Aptología (2026), datos abiertos, licencia CC BY 4.0, corte 22/11/2026. Este análisis es exploratorio y no constituye avalúo ni recomendación de renta.

> Ejecuta las celdas en orden. Para usar los filtros interactivos, abre el notebook en Jupyter o Google Colab y ejecuta `pip install -r requirements.txt`.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 30)
pd.options.display.float_format = '{:,.2f}'.format

ROOT = Path.cwd()
if not (ROOT / 'data/mx-renta-por-colonia-2026-11.csv').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data/mx-renta-por-colonia-2026-11.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError('No encuentro data/mx-renta-por-colonia-2026-11.csv. Ejecuta el notebook desde la carpeta del proyecto.')

raw = pd.read_csv(DATA_PATH)
required = {'pais','ciudad','zona','operacion','tipo','avisos','mediana','p25','p75','mediana_area_m2'}
missing_columns = required - set(raw.columns)
if missing_columns:
    raise ValueError(f'Faltan columnas necesarias: {sorted(missing_columns)}')
print(f'Archivo cargado: {DATA_PATH.name} · {len(raw):,} filas · {raw.shape[1]} columnas')

Archivo cargado: mx-renta-por-colonia-2026-11.csv · 10,179 filas · 13 columnas


## 1. Calidad y preparación

Primero inspeccionamos faltantes, duplicados y tipos. Los valores atípicos se **marcan** con la regla 1.5×IQR para investigarlos; se conservan porque las filas son medianas de segmentos y sin los anuncios fuente no es posible afirmar que sean errores.

In [2]:
numeric = ['avisos','mediana','p25','p75','mediana_m2','mediana_area_m2']
for col in numeric:
    if col in raw:
        raw[col] = pd.to_numeric(raw[col], errors='coerce')

quality = pd.DataFrame({
    'filas_con_faltante': raw.isna().sum(),
    'porcentaje_faltante': (raw.isna().mean() * 100).round(2),
    'tipo_dato': raw.dtypes.astype(str)
}).sort_values('filas_con_faltante', ascending=False)
display(quality)

key = ['pais','ciudad','zona','operacion','tipo']
duplicate_rows = int(raw.duplicated(key, keep='first').sum())
print(f'Registros duplicados según {key}: {duplicate_rows:,}')
display(raw.describe(include='all').T)

,filas_con_faltante,porcentaje_faltante,tipo_dato
mediana_area_m2,470,4.62,float64
mediana_m2,470,4.62,float64
zona,0,0.00,str
ciudad,0,0.00,str
pais,0,0.00,str
tipo,0,0.00,str
operacion,0,0.00,str
avisos,0,0.00,int64
mediana,0,0.00,int64
p75,0,0.00,int64


Registros duplicados según ['pais', 'ciudad', 'zona', 'operacion', 'tipo']: 0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
pais,10179,1,mx,10179,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ciudad,10179,562,Ciudad de México,1864,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zona,10179,3150,Centro,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
operacion,10179,2,venta,7912,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tipo,10179,3,todos,4752,NaN,NaN,NaN,NaN,NaN,NaN,NaN
avisos,"10,179.00",NaN,NaN,NaN,46.42,232.62,5.00,6.00,11.00,25.00,"8,140.00"
mediana,"10,179.00",NaN,NaN,NaN,"5,886,914.79","7,514,992.54","4,990.00","1,322,250.00","4,050,000.00","7,340,000.00","98,837,113.00"
p25,"10,179.00",NaN,NaN,NaN,"4,394,773.13","5,491,451.97","3,150.00","940,000.00","3,154,200.00","5,500,000.00","83,857,260.00"
p75,"10,179.00",NaN,NaN,NaN,"8,178,937.85","10,609,990.53","5,500.00","1,815,000.00","5,345,000.00","10,143,742.00","126,420,500.00"
mediana_m2,"9,709.00",NaN,NaN,NaN,"29,411.54","24,477.21",34.00,"13,861.00","25,783.00","42,737.00","375,000.00"


## 2. Filtros exploratorios

Los selectores filtran ciudad, operación, tipo, texto de zona, mínimo de anuncios y rango de renta. La visualización EDA y las tablas de esta sección se recalculan con la selección. Los filtros se aplican sobre la fuente completa; el modelado posterior mantiene el alcance fijo de Fase 2 para que las métricas sean comparables.

In [3]:
# Normalizamos textos y construimos una copia analítica sin eliminar todavía filas.
data = raw.copy()
for col in ['pais','ciudad','zona','operacion','tipo']:
    data[col] = data[col].astype('string').str.strip()
data['mediana'] = pd.to_numeric(data['mediana'], errors='coerce')
data['mediana_area_m2'] = pd.to_numeric(data['mediana_area_m2'], errors='coerce')
data['avisos'] = pd.to_numeric(data['avisos'], errors='coerce')
data['p25'] = pd.to_numeric(data['p25'], errors='coerce')
data['p75'] = pd.to_numeric(data['p75'], errors='coerce')
data = data.dropna(subset=['ciudad','zona','operacion','tipo','mediana','mediana_area_m2','avisos'])
data = data[(data['mediana'] > 0) & (data['mediana_area_m2'] > 0) & (data['avisos'] >= 1)].copy()

try:
    import ipywidgets as widgets
    from IPython.display import display as ipy_display
    all_option = 'Todos'
    cities = [all_option] + sorted(data['ciudad'].dropna().unique().tolist())
    operations = [all_option] + sorted(data['operacion'].dropna().unique().tolist())
    types = [all_option] + sorted(data['tipo'].dropna().unique().tolist())
    min_rent, max_rent = int(data.mediana.min()), int(data.mediana.max())
    import unicodedata
    city_match = next((c for c in cities if c != all_option and unicodedata.normalize('NFKD', str(c)).encode('ascii','ignore').decode('ascii').lower() == 'ciudad de mexico'), all_option)
    city_w = widgets.Dropdown(options=cities, value=city_match, description='Ciudad:', layout={'width':'360px'})
    operation_w = widgets.Dropdown(options=operations, value='alquiler' if 'alquiler' in operations else all_option, description='Operación:', layout={'width':'300px'})
    type_w = widgets.Dropdown(options=types, value=all_option, description='Tipo:', layout={'width':'300px'})
    zone_w = widgets.Text(value='', description='Zona contiene:', placeholder='Ej. Roma', layout={'width':'360px'})
    avisos_w = widgets.IntSlider(value=5, min=1, max=max(5, int(data.avisos.quantile(.95))), step=1, description='Mín. avisos:', continuous_update=False, layout={'width':'500px'})
    rent_w = widgets.IntRangeSlider(value=(min_rent, max_rent), min=min_rent, max=max_rent, step=max(1, (max_rent-min_rent)//200), description='Renta MXN:', continuous_update=False, layout={'width':'650px'})
    filter_out = widgets.Output()
    def apply_filters(_=None):
        selected = data.copy()
        if city_w.value != all_option: selected = selected[selected.ciudad == city_w.value]
        if operation_w.value != all_option: selected = selected[selected.operacion == operation_w.value]
        if type_w.value != all_option: selected = selected[selected.tipo == type_w.value]
        if zone_w.value.strip(): selected = selected[selected.zona.str.contains(zone_w.value.strip(), case=False, na=False)]
        selected = selected[(selected.avisos >= avisos_w.value) & selected.mediana.between(*rent_w.value)]
        globals()['filtered'] = selected.copy()
        with filter_out:
            filter_out.clear_output(wait=True)
            print(f'{len(selected):,} filas · {selected.zona.nunique():,} zonas · renta {rent_w.value[0]:,}–{rent_w.value[1]:,} MXN · mínimo {avisos_w.value} avisos')
            if selected.empty: print('No hay filas con esta combinación. Amplía el rango o cambia los filtros.')
    for w in [city_w, operation_w, type_w, zone_w, avisos_w, rent_w]:
        w.observe(apply_filters, names='value')
    ipy_display(widgets.VBox([widgets.HBox([city_w, operation_w]), widgets.HBox([type_w, zone_w]), avisos_w, rent_w, filter_out]))
    apply_filters()
except ImportError:
    # Fallback editable sin ipywidgets: cambia estos valores y vuelve a ejecutar esta celda.
    city_filter = 'Ciudad de Mexico'  # 'Todos' o nombre de ciudad
    operation_filter = 'alquiler'     # 'Todos', alquiler, venta, etc.
    type_filter = 'Todos'             # 'Todos', casa, departamento, etc.
    zone_contains = ''                # Ejemplo: 'Roma'
    min_ads, rent_min, rent_max = 5, 0, float('inf')
    filtered = data.copy()
    if city_filter != 'Todos': filtered = filtered[filtered.ciudad == city_filter]
    if operation_filter != 'Todos': filtered = filtered[filtered.operacion == operation_filter]
    if type_filter != 'Todos': filtered = filtered[filtered.tipo == type_filter]
    if zone_contains: filtered = filtered[filtered.zona.str.contains(zone_contains, case=False, na=False)]
    filtered = filtered[(filtered.avisos >= min_ads) & filtered.mediana.between(rent_min, rent_max)]
    print('ipywidgets no está instalado; ajusta las variables de filtro de esta celda.')
    print(f'{len(filtered):,} filas · {filtered.zona.nunique():,} zonas')

## 3. Resumen de la selección y tablas descriptivas

Lee las tablas con los filtros activos. Las columnas `n_filas` y `avisos` ayudan a no comparar medianas sin considerar cuánta cobertura hay detrás de cada segmento.

In [4]:
if filtered.empty:
    print('La selección está vacía; cambia los filtros antes de continuar.')
else:
    kpis = pd.DataFrame([
        ['Filas agregadas', len(filtered)], ['Zonas', filtered.zona.nunique()],
        ['Tipos', filtered.tipo.nunique()], ['Anuncios representados (suma)', int(filtered.avisos.sum())],
        ['Renta mediana (MXN)', filtered.mediana.median()], ['Superficie mediana (m²)', filtered.mediana_area_m2.median()]
    ], columns=['Indicador','Valor'])
    display(kpis.style.format({'Valor': lambda x: f'{x:,.0f}' if isinstance(x,(int,float,np.integer,np.floating)) else x}).set_caption('Indicadores de la selección'))

    tabla_tipo = (filtered.groupby('tipo').agg(n_filas=('zona','size'), zonas=('zona','nunique'), avisos=('avisos','sum'),
        renta_mediana=('mediana','median'), renta_promedio=('mediana','mean'), renta_p25=('mediana','quantile'),
        superficie_mediana=('mediana_area_m2','median')).sort_values('renta_mediana', ascending=False).reset_index())
    display(tabla_tipo.style.format({'avisos':'{:,.0f}','renta_mediana':'${:,.0f}','renta_promedio':'${:,.0f}','renta_p25':'${:,.0f}','superficie_mediana':'{:,.1f}'}).set_table_styles([{'selector':'th','props':[('background-color','#17324D'),('color','white')]}]).set_caption('Resumen por tipo de inmueble'))

    tabla_zona = (filtered.groupby('zona').agg(n_filas=('tipo','size'), tipos=('tipo','nunique'), avisos=('avisos','sum'),
        renta_mediana=('mediana','median'), renta_p25=('mediana','quantile'), renta_p75=('mediana',lambda x:x.quantile(.75)),
        superficie_mediana=('mediana_area_m2','median')).sort_values('renta_mediana',ascending=False).reset_index())
    display(tabla_zona.head(15).style.format({'avisos':'{:,.0f}','renta_mediana':'${:,.0f}','renta_p25':'${:,.0f}','renta_p75':'${:,.0f}','superficie_mediana':'{:,.1f}'}).set_table_styles([{'selector':'th','props':[('background-color','#168AAD'),('color','white')]}]).set_caption('15 zonas con renta mediana más alta en la selección'))

,Indicador,Valor
0,Filas agregadas,476
1,Zonas,228
2,Tipos,3
3,Anuncios representados (suma),"15,314"
4,Renta mediana (MXN),"29,550"
5,Superficie mediana (m²),100


,tipo,n_filas,zonas,avisos,renta_mediana,renta_promedio,renta_p25,superficie_mediana
0,casa,61,61,919,"$73,000","$85,497","$73,000",350.0
1,todos,228,228,"7,839","$28,700","$36,466","$28,700",98.5
2,departamento,187,187,"6,556","$26,000","$31,435","$26,000",85.0


,zona,n_filas,tipos,avisos,renta_mediana,renta_p25,renta_p75,superficie_mediana
0,Lomas De Reforma,1,1,7,"$135,000","$135,000","$135,000",468.0
1,Insurgentes San Borja,2,2,12,"$130,000","$130,000","$130,000",317.0
2,Lomas De Chapultepec VIII Sección,1,1,5,"$120,000","$120,000","$120,000",540.0
3,Bosque De Las Lomas,3,3,90,"$120,000","$120,000","$125,000",479.0
4,Lomas De Chapultepec,3,3,92,"$110,000","$110,000","$160,000",370.0
5,Lomas De Chapultepec III Sección,2,2,17,"$109,750","$109,750","$114,625",295.0
6,Jardines En La Montaña,3,3,64,"$100,000","$100,000","$106,250",470.0
7,San Mateo Tlaltenango,2,2,16,"$97,500","$97,500","$103,750",260.0
8,Bosques de las Lomas,3,3,100,"$97,500","$97,500","$115,000",410.0
9,Lomas de Chapultepec I Sección,2,2,24,"$93,750","$93,750","$96,875",258.0


## 4. Visualizaciones EDA

Los gráficos son interactivos (hover, zoom y leyendas). La distribución y los diagramas de caja muestran dispersión; el scatter compara medianas de zona y tipo. No se interpretan como relaciones causales.

In [5]:
if not filtered.empty:
    fig = px.histogram(filtered, x='mediana', color='tipo', nbins=30, marginal='box',
        title='Distribución de rentas medianas por zona y tipo', labels={'mediana':'Renta mensual mediana (MXN)','count':'Número de celdas','tipo':'Tipo'}, opacity=.78)
    fig.update_layout(bargap=.08, legend_title_text='Tipo de inmueble'); fig.show()

    fig = px.box(filtered, x='tipo', y='mediana', points='outliers', color='tipo',
        title='Dispersión de la renta mediana por tipo', labels={'tipo':'Tipo','mediana':'Renta mensual mediana (MXN)'}, hover_data=['zona','avisos','mediana_area_m2'])
    fig.show()

    fig = px.scatter(filtered, x='mediana_area_m2', y='mediana', color='tipo', size='avisos', size_max=18,
        hover_name='zona', hover_data={'avisos':True,'p25':True,'p75':True,'mediana_area_m2':':.1f'},
        title='Renta y superficie medianas (tamaño del punto = anuncios representados)',
        labels={'mediana_area_m2':'Superficie mediana (m²)','mediana':'Renta mensual mediana (MXN)','tipo':'Tipo'})
    fig.show()

    by_type = filtered.groupby('tipo',as_index=False).agg(renta=('mediana','median'), superficie=('mediana_area_m2','median'), avisos=('avisos','sum'))
    fig = px.scatter(by_type, x='superficie', y='renta', size='avisos', color='tipo', text='tipo',
        title='Comparación multivariada de medianas por tipo', labels={'superficie':'Superficie mediana (m²)','renta':'Renta mediana (MXN)','avisos':'Anuncios'})
    fig.update_traces(textposition='top center'); fig.show()

## 5. Comparaciones, correlación y observaciones atípicas

La matriz de correlación resume asociaciones lineales entre variables numéricas de la selección. Las observaciones IQR se conservan y se muestran para revisión contextual.

In [6]:
if not filtered.empty:
    corr_cols = [c for c in ['mediana','mediana_area_m2','avisos','p25','p75','mediana_m2'] if c in filtered and filtered[c].nunique()>1]
    corr = filtered[corr_cols].corr(numeric_only=True)
    fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1, aspect='auto',
        title='Correlaciones entre medidas numéricas (Pearson)', labels={'color':'Correlación'})
    fig.show()

    q1,q3=filtered.mediana.quantile([.25,.75]); iqr=q3-q1; low=max(0,q1-1.5*iqr); high=q3+1.5*iqr
    reviewed=filtered[(filtered.mediana<low)|(filtered.mediana>high)].copy().sort_values('mediana',ascending=False)
    print(f'Regla 1.5×IQR en la selección: límites ${low:,.0f}–${high:,.0f}; filas señaladas {len(reviewed)} de {len(filtered)}. Se conservan, no se eliminan automáticamente.')
    if not reviewed.empty:
        display(reviewed[['ciudad','zona','operacion','tipo','avisos','mediana','p25','p75','mediana_area_m2']].head(30).style.format({'mediana':'${:,.0f}','p25':'${:,.0f}','p75':'${:,.0f}','mediana_area_m2':'{:,.1f}'}).set_caption('Filas señaladas para revisión contextual'))
    else:
        print('No hay filas fuera de los límites IQR con esta selección.')

Regla 1.5×IQR en la selección: límites $0–$82,781; filas señaladas 45 de 476. Se conservan, no se eliminan automáticamente.


,ciudad,zona,operacion,tipo,avisos,mediana,p25,p75,mediana_area_m2
2382,Ciudad de México,Polanco IV Sección,alquiler,casa,18,"$235,000","$120,000","$295,000",450.0
1977,Ciudad de México,Lomas De Chapultepec,alquiler,casa,21,"$210,000","$195,000","$250,000",750.0
2376,Ciudad de México,Polanco III Sección,alquiler,casa,19,"$198,000","$122,500","$250,000",326.0
1976,Ciudad de México,Lomas de Chapultepec,alquiler,casa,36,"$188,500","$100,000","$300,000",600.0
1822,Ciudad de México,Jardines del Pedregal,alquiler,casa,10,"$157,500","$121,250","$176,250",559.0
1189,Ciudad de México,Anzures,alquiler,casa,28,"$152,500","$72,250","$219,750",512.0
2388,Ciudad de México,Polanco V Sección,alquiler,casa,8,"$150,000","$131,250","$201,935",400.0
1271,Ciudad de México,Bosques De Las Lomas,alquiler,casa,16,"$137,500","$88,000","$160,000",530.0
2648,Ciudad de México,San Rafael,alquiler,casa,8,"$135,000","$31,399","$215,000",420.0
2028,Ciudad de México,Lomas De Reforma,alquiler,todos,7,"$135,000","$62,500","$150,000",468.0


## 6. Modelo analítico de referencia (alcance fijo de Fase 2)

Para evitar mezclar ciudades y operaciones, el modelo se restringe a Ciudad de México, alquiler, casa/departamento. Se excluyen faltantes indispensables, valores no positivos y segmentos con menos de cinco anuncios; se eliminan duplicados por zona/tipo/operación. Las zonas de prueba se reservan completas para reducir fuga territorial.

In [7]:
city_norm = raw.ciudad.astype(str).str.normalize('NFKD').str.encode('ascii',errors='ignore').str.decode('ascii').str.lower()
model_raw = raw[(raw.pais == 'mx') & city_norm.eq('ciudad de mexico') & (raw.operacion == 'alquiler') & raw.tipo.isin(['casa','departamento'])].copy()
scope_initial = len(model_raw)
model = model_raw.dropna(subset=['zona','tipo','mediana','mediana_area_m2','avisos'])
model = model[(model.mediana>0)&(model.mediana_area_m2>0)&(model.avisos>=5)].drop_duplicates(['zona','tipo','operacion']).copy()
q1,q3=model.mediana.quantile([.25,.75]); lo=q1-1.5*(q3-q1); hi=q3+1.5*(q3-q1)
model['atipico_iqr']=(model.mediana<lo)|(model.mediana>hi)
model['log_area']=np.log(model.mediana_area_m2); model['log_renta']=np.log(model.mediana)
scrub_table = pd.DataFrame([
    ['Filas iniciales en alcance',scope_initial],['Filas limpias',len(model)],['Zonas limpias',model.zona.nunique()],
    ['Faltantes de superficie excluidos',int(model_raw.mediana_area_m2.isna().sum())],
    ['Duplicados de llave eliminados',int(model_raw.duplicated(['zona','tipo','operacion']).sum())],
    ['Valores IQR señalados y conservados',int(model.atipico_iqr.sum())]
],columns=['Regla/medida','Resultado'])
display(scrub_table)
display(model.groupby('tipo').agg(n=('zona','size'),zonas=('zona','nunique'),avisos=('avisos','sum'),renta_mediana=('mediana','median'),area_mediana=('mediana_area_m2','median')).reset_index())

,Regla/medida,Resultado
0,Filas iniciales en alcance,256
1,Filas limpias,248
2,Zonas limpias,197
3,Faltantes de superficie excluidos,8
4,Duplicados de llave eliminados,0
5,Valores IQR señalados y conservados,17


,tipo,n,zonas,avisos,renta_mediana,area_mediana
0,casa,61,61,919,"73,000.00",350.00
1,departamento,187,187,6556,"26,000.00",85.00


In [8]:
zones=sorted(model.zona.unique())
test_z=set(zones[::5])
train=model[~model.zona.isin(test_z)].copy(); test=model[model.zona.isin(test_z)].copy()
mu=train.log_area.mean(); sd=train.log_area.std() or 1
def design(frame):
    return np.column_stack([np.ones(len(frame)),(frame.log_area.to_numpy()-mu)/sd,(frame.tipo.to_numpy()=='casa').astype(float)])
beta=np.linalg.lstsq(design(train),train.log_renta.to_numpy(),rcond=None)[0]
y=test.mediana.to_numpy(); pred=np.exp(design(test)@beta)
mae=float(np.mean(np.abs(y-pred))); rmse=float(np.sqrt(np.mean((y-pred)**2)))
r2=float(1-np.sum((y-pred)**2)/np.sum((y-y.mean())**2))
base=float(np.mean(np.abs(y-train.mediana.median())))
metrics=pd.DataFrame([
    ['R²',r2,'Proporción de variación en la prueba; puede ser sensible al tamaño y composición de la muestra.'],
    ['MAE (MXN)',mae,'Error absoluto medio frente a rentas medianas agregadas.'],
    ['RMSE (MXN)',rmse,'Penaliza más errores grandes.'],
    ['MAE base (MXN)',base,'Predicción constante con mediana del conjunto de entrenamiento.'],
    ['Mejora MAE vs base (%)',(1-mae/base)*100,'Comparación dentro de esta única partición de prueba.']
],columns=['Métrica','Valor','Interpretación'])
display(metrics.style.format({'Valor':lambda x:f'{x:.3f}' if abs(x)<1 else f'{x:,.0f}'}).set_caption(f'OLS · entrenamiento {len(train)} filas · prueba {len(test)} filas · zonas de prueba {len(test_z)}'))

pred_test=test[['zona','tipo','mediana','mediana_area_m2','avisos']].copy()
pred_test['prediccion_mxn']=pred; pred_test['error_mxn']=pred_test.mediana-pred_test.prediccion_mxn
display(pred_test.assign(error_abs=pred_test.error_mxn.abs()).sort_values('error_abs',ascending=False).head(15).style.format({'mediana':'${:,.0f}','prediccion_mxn':'${:,.0f}','error_mxn':'${:,.0f}','error_abs':'${:,.0f}','mediana_area_m2':'{:,.1f}'}).set_caption('15 errores absolutos más altos en prueba; revisar como diagnóstico'))

,Métrica,Valor,Interpretación
0,R²,0.796,Proporción de variación en la prueba; puede ser sensible al tamaño y composición de la muestra.
1,MAE (MXN),"8,907",Error absoluto medio frente a rentas medianas agregadas.
2,RMSE (MXN),"13,139",Penaliza más errores grandes.
3,MAE base (MXN),"20,130",Predicción constante con mediana del conjunto de entrenamiento.
4,Mejora MAE vs base (%),56,Comparación dentro de esta única partición de prueba.


,zona,tipo,mediana,mediana_area_m2,avisos,prediccion_mxn,error_mxn,error_abs
2394,Polanco,casa,"$133,000",378.0,33,"$80,574","$52,426","$52,426"
1481,Condesa,casa,"$95,000",290.0,11,"$62,324","$32,676","$32,676"
2621,San Nicolás Totolapan,casa,"$42,500",319.0,7,"$68,355","$-25,855","$25,855"
1937,Lomas De Chapultepec I Sección,departamento,"$50,000",260.0,5,"$73,573","$-23,573","$23,573"
2393,Polanco,departamento,"$70,000",170.0,147,"$48,741","$21,259","$21,259"
1749,Héroes De Padierna,casa,"$44,250",304.0,6,"$65,238","$-20,988","$20,988"
2078,Lomas Del Chamizal,departamento,"$89,500",262.0,36,"$74,121","$15,379","$15,379"
1852,Juárez,departamento,"$34,500",65.0,284,"$19,198","$15,302","$15,302"
2084,Lomas del Pedregal Framboyanes,departamento,"$72,000",199.0,8,"$56,778","$15,222","$15,222"
1480,Condesa,departamento,"$39,750",85.0,144,"$24,898","$14,852","$14,852"


In [9]:
fig = px.scatter(pred_test, x='mediana', y='prediccion_mxn', color='tipo', size='avisos', hover_name='zona',
    title='Rentas observadas y predichas en zonas reservadas', labels={'mediana':'Renta observada (MXN)','prediccion_mxn':'Predicción del modelo (MXN)','tipo':'Tipo'})
lo=min(pred_test.mediana.min(),pred_test.prediccion_mxn.min()); hi=max(pred_test.mediana.max(),pred_test.prediccion_mxn.max())
fig.add_trace(go.Scatter(x=[lo,hi],y=[lo,hi],mode='lines',name='Referencia ideal',line={'dash':'dash','color':'gray'})); fig.show()

pred_test['residuo']=pred_test.mediana-pred_test.prediccion_mxn
fig=px.scatter(pred_test,x='prediccion_mxn',y='residuo',color='tipo',hover_name='zona',
    title='Diagnóstico de residuos',labels={'prediccion_mxn':'Renta predicha (MXN)','residuo':'Residuo: observada − predicha (MXN)'})
fig.add_hline(y=0,line_dash='dash',line_color='gray'); fig.show()

## 7. Lectura de resultados, límites y siguientes pasos

El modelo se ajusta a medianas de segmentos zona-tipo, no a viviendas individuales. R², MAE y RMSE corresponden a una sola partición por colonia; no garantizan desempeño futuro ni en zonas con otra cobertura. Las rentas son precios anunciados, no cierres. Antes de usar cualquier cifra para decisiones, haría falta ampliar y verificar datos, validar por periodo y territorio y contar con revisión humana.

### Evidencias que debes comentar

1. Indica los filtros activos y cuántas filas/zonas quedaron.
2. Describe mediana y rango intercuartílico por tipo junto con anuncios representados.
3. Explica qué patrones muestran histograma, caja, dispersión y tabla de zonas; evita afirmar causalidad.
4. Compara el modelo con la línea base y presenta MAE/RMSE en MXN.
5. Reporta como limitación el origen de portales, los agregados, el tamaño de prueba y el dashboard aún local.

**Referencia de datos:** Aptología. (2026). *Datos abiertos de rentas en México* (corte 22 de noviembre de 2026) [Conjunto de datos, CC BY 4.0]. https://www.aptologia.com/mx/datos-abiertos